CUSTOMER ORDERS ANALYSIS WITH PYSPARK

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerOrdersLab") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 09:59:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/02 09:59:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/02 09:59:06 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/07/02 09:59:06 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
from pyspark.sql.functions import (
    col, to_date, sum, count,
    year, month, dayofmonth
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType
)

In [7]:
dataschema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("unit_price", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("payment_mode", StringType(), True),
    StructField("city", StringType(), True)
])

In [10]:
data = """order_id,order_date,customer_id,customer_name,product_category,product_name,brand,unit_price,quantity,payment_mode,city
1,2026-01-01,C001,Rahul Sharma,Electronics,Mobile,Apple,50000,1,UPI,Mumbai
2,2026-01-02,C002,Anita Patel,Electronics,Laptop,Dell,60000,1,Credit Card,Pune
3,2026-01-03,C003,Amit Verma,Home Appliances,Mixer,Philips,3000,2,Cash,Delhi
4,2026-01-04,C001,Rahul Sharma,Fashion,T-Shirt,Nike,1500,3,UPI,Mumbai
5,2026-01-05,C004,Priya Singh,Electronics,Headphones,Sony,2000,2,Debit Card,Bangalore
6,2026-01-06,C005,John Doe,Fashion,Shoes,Puma,2500,1,Credit Card,Hyderabad
7,2026-02-01,C002,Anita Patel,Home Appliances,Microwave,LG,12000,1,UPI,Pune
8,2026-02-03,C006,Neha Gupta,Electronics,Tablet,Samsung,25000,1,Cash,Delhi
9,2026-02-05,C007,Vikram Rao,Fashion,Jeans,Levis,3000,2,UPI,Mumbai
10,2026-02-07,C008,Sneha Iyer,Home Appliances,Fridge,Whirlpool,20000,1,Debit Card,Chennai
"""

In [12]:
file_path = "/home/kasm-user/Desktop/Downloads/spark/data.csv"

with open(file_path, "w") as f:
    f.write(data)

In [13]:
df = spark.read \
    .option("header", "true") \
    .schema(dataschema) \
    .csv("/home/kasm-user/Downloads/spark/data.csv")

In [14]:
df.printSchema()
df.show()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- city: string (nullable = true)

+--------+----------+-----------+-------------+----------------+------------+---------+----------+--------+------------+---------+
|order_id|order_date|customer_id|customer_name|product_category|product_name|    brand|unit_price|quantity|payment_mode|     city|
+--------+----------+-----------+-------------+----------------+------------+---------+----------+--------+------------+---------+
|       1|2026-01-01|       C001| Rahul Sharma|     Electronics|      Mobile|    Apple|     50000|       1|         UPI|   Mumbai|
|     

In [15]:
df1 = df.withColumn(
    "order_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)


In [16]:
df2 = df1.withColumn(
    "total_amount",
    col("unit_price") * col("quantity")
)

In [17]:
df3 = df2.withColumnRenamed("unit_price", "price")

In [18]:
df4 = df3.withColumn(
    "order_year",
    year(col("order_date"))
)


In [19]:
df5 = df4.withColumn(
    "order_month",
    month(col("order_date"))
)

In [20]:
df6 = df5.withColumn(
    "order_day",
    dayofmonth(col("order_date"))
)

In [21]:
category_sales = df6.groupBy("product_category") \
    .agg(sum("total_amount").alias("total_sales"))

In [22]:
category_sales.orderBy(col("total_sales").desc()).show()

+----------------+-----------+
|product_category|total_sales|
+----------------+-----------+
|     Electronics|     139000|
| Home Appliances|      38000|
|         Fashion|      13000|
+----------------+-----------+



In [23]:
brand_orders = df6.groupBy("brand") \
    .agg(count("order_id").alias("order_count"))

brand_orders.show()

+---------+-----------+
|    brand|order_count|
+---------+-----------+
|    Levis|          1|
|     Nike|          1|
|     Sony|          1|
|     Dell|          1|
|     Puma|          1|
|  Philips|          1|
|  Samsung|          1|
|       LG|          1|
|    Apple|          1|
|Whirlpool|          1|
+---------+-----------+



In [24]:
city_sales = df6.groupBy("city") \
    .agg(sum("total_amount").alias("city_sales"))


In [25]:
city_sales.orderBy(col("city_sales").desc()).show()

+---------+----------+
|     city|city_sales|
+---------+----------+
|     Pune|     72000|
|   Mumbai|     60500|
|    Delhi|     31000|
|  Chennai|     20000|
|Bangalore|      4000|
|Hyderabad|      2500|
+---------+----------+



In [27]:
payment_sales = df6.groupBy("payment_mode") \
    .agg(sum("total_amount").alias("payment_sales"))
payment_sales.show()




+------------+-------------+
|payment_mode|payment_sales|
+------------+-------------+
| Credit Card|        62500|
|        Cash|        31000|
|  Debit Card|        24000|
|         UPI|        72500|
+------------+-------------+



In [28]:

monthly_orders = df6.groupBy("order_month") \
    .agg(count("order_id").alias("total_orders"))

monthly_orders.show()


+-----------+------------+
|order_month|total_orders|
+-----------+------------+
|          1|           6|
|          2|           4|
+-----------+------------+



In [29]:
final_df = df6.select(
    "order_id","order_date","customer_name",
    "product_category","product_name","brand",
    "price","quantity","total_amount",
    "payment_mode","city"
)

final_df.show(truncate=False)

+--------+----------+-------------+----------------+------------+---------+-----+--------+------------+------------+---------+
|order_id|order_date|customer_name|product_category|product_name|brand    |price|quantity|total_amount|payment_mode|city     |
+--------+----------+-------------+----------------+------------+---------+-----+--------+------------+------------+---------+
|1       |2026-01-01|Rahul Sharma |Electronics     |Mobile      |Apple    |50000|1       |50000       |UPI         |Mumbai   |
|2       |2026-01-02|Anita Patel  |Electronics     |Laptop      |Dell     |60000|1       |60000       |Credit Card |Pune     |
|3       |2026-01-03|Amit Verma   |Home Appliances |Mixer       |Philips  |3000 |2       |6000        |Cash        |Delhi    |
|4       |2026-01-04|Rahul Sharma |Fashion         |T-Shirt     |Nike     |1500 |3       |4500        |UPI         |Mumbai   |
|5       |2026-01-05|Priya Singh  |Electronics     |Headphones  |Sony     |2000 |2       |4000        |Debit Ca

In [31]:
spark.stop()